# RF Amplifier Compression

Simulation of an RF amplifier with gain compression demonstrating the third-order nonlinearity model. We sweep the input power and observe the 1 dB compression point and gain saturation behavior.

## Amplifier Model

The `RFAmplifier` block implements a third-order polynomial nonlinearity:

$$y(t) = a_1 x(t) + a_3 x^3(t)$$

where $a_1$ is the linear voltage gain and $a_3$ is derived from the input-referred third-order intercept point (IIP3). The output is hard-clipped at the gain compression peak to prevent unphysical sign reversal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply PathSim docs matplotlib style
plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope, Spectrum
from pathsim.solvers import RKCK54

from pathsim_rf import RFAmplifier

## System Setup

We create an amplifier with 20 dB gain and an IIP3 of +10 dBm, and drive it with a single-tone sinusoidal source at 1 GHz. We sweep the input amplitude to trace out the compression curve.

In [ ]:
# Amplifier parameters
gain_dB = 20.0    # Small-signal gain [dB]
IIP3_dBm = 10.0   # Input-referred IP3 [dBm]
Z0 = 50.0         # Reference impedance [Ohm]
f0 = 100.0        # Signal frequency [Hz] (scaled for simulation)

# Sweep input power levels
pin_dbm = np.arange(-30, 15, 1.0)
pout_dbm = np.zeros_like(pin_dbm)

for i, p_in in enumerate(pin_dbm):

    # Input amplitude from power in dBm
    p_watts = 10.0 ** (p_in / 10.0) * 1e-3
    v_peak = np.sqrt(2.0 * Z0 * p_watts)

    # Build simulation
    src = Source(func=lambda t, vp=v_peak: vp * np.sin(2 * np.pi * f0 * t))
    amp = RFAmplifier(gain=gain_dB, IIP3=IIP3_dBm, Z0=Z0)
    sco = Scope()

    sim = Simulation(
        [src, amp, sco],
        [Connection(src, amp), Connection(amp, sco)],
        dt=1 / (40 * f0),
        Solver=RKCK54
    )

    # Run for several cycles to reach steady state
    sim.run(10 / f0)

    # Read output and measure peak amplitude (last 2 cycles)
    t, [y] = sco.read()
    n_last = int(len(t) * 0.5)
    v_out_peak = np.max(np.abs(y[n_last:]))

    # Convert to output power in dBm
    p_out = v_out_peak ** 2 / (2 * Z0)
    pout_dbm[i] = 10 * np.log10(p_out / 1e-3) if p_out > 0 else -100

## Compression Curve

The plot shows output power vs. input power. The dashed line represents ideal linear gain. The deviation from linearity shows the gain compression characteristic.

In [ ]:
fig, ax = plt.subplots(dpi=120)

# Ideal linear response
ax.plot(pin_dbm, pin_dbm + gain_dB, '--', label='Ideal (linear)', alpha=0.7)

# Simulated response
ax.plot(pin_dbm, pout_dbm, linewidth=2, label='Simulated')

# Mark P1dB point
p1db_in = IIP3_dBm - 9.6
ax.axvline(x=p1db_in, color='grey', linestyle=':', alpha=0.5, label=f'P1dB = {p1db_in:.1f} dBm')

ax.set_xlabel('Input Power [dBm]')
ax.set_ylabel('Output Power [dBm]')
ax.set_title('RF Amplifier Compression Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Time-Domain Waveforms

Let's compare the output waveform in the linear and compressed regimes to visualize the clipping behavior.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120)

for ax, p_in, title in zip(axes, [-20, 5], ['Linear regime (-20 dBm)', 'Compressed regime (+5 dBm)']):
    p_watts = 10.0 ** (p_in / 10.0) * 1e-3
    v_peak = np.sqrt(2.0 * Z0 * p_watts)

    src = Source(func=lambda t, vp=v_peak: vp * np.sin(2 * np.pi * f0 * t))
    amp = RFAmplifier(gain=gain_dB, IIP3=IIP3_dBm, Z0=Z0)
    sco_in = Scope(labels=['input'])
    sco_out = Scope(labels=['output'])

    sim = Simulation(
        [src, amp, sco_in, sco_out],
        [Connection(src, amp, sco_in), Connection(amp, sco_out)],
        dt=1 / (40 * f0),
        Solver=RKCK54
    )

    sim.run(3 / f0)

    t_in, [y_in] = sco_in.read()
    t_out, [y_out] = sco_out.read()

    ax.plot(np.array(t_in) * f0, y_in * 1000, label='Input')
    ax.plot(np.array(t_out) * f0, y_out * 1000, label='Output')
    ax.set_xlabel('Cycles')
    ax.set_ylabel('Voltage [mV]')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()